## Tokenization

原始的文本通常是Unicode 字符串，tokenization是将字符串变成一个整数的序列，其中的每个整数为一个token

需要一个程序完成字符串到token的编码转换，同时也需要其从token解码回字符串，Tokenizer就是实现编码解码方法的类

词典的大小(vocabulary size)是tokens的所有可能取值

需要注意的是**空格的位置space**也是token的一部分，比如"hello "和" hello"经过tokenizer编码之后的整数值不同

数字也会经过某种语义的规则转变成整数的列表，gpt-4o的tokenizer会把12345678，切分为123、456、78并分别给三段进行编码

一个简单的使用tokenizer的编码和解码规则的程序片段

```python
tokenizer = get_gpt2_tokenizer()
string = "Hello, 🌏! 你好！"

indices = tokenizer.encode(string)
reconstructed_string = tokenizer.decode(indices)
assert string == reconstructed_string
compression_ratio = get_compression_ratio(string, indices) # numsOfBytes / numsOfTokens=1.667
```

tokenizer实现的最简单的思路是**Character-based tokenization**

一个Unicode字符串由一个Unicode的字符序列构成，每个字符可以被转化为一个整数

这种tokenizer的问题在于，每个char都要有一个映射，实际上在字符串中一些char的出现频率很高，另一些的出现频率很低，这会导致vocabulary的规模很大且利用的效率很低

另一种方式为**Byte-based tokenization**

Unicode字符串可以表示为一个字节序列，因为每个字符串都可以转换为字节，例如

```python
# using UTF-8
assert bytes("a", encoding="utf-8") == b"a"
assert bytes("🌏", encoding="utf-8") == b"\xf0\x9f\x8c\x8d"
```

则字符串可以转化为0-255之间的整数，最常用的Unicode编码为**UTF-8**

```python
tokenizer = ByteTokenizer()
string = "Hello, 🌏! 你好！"
indices = tokenizer.encode(string)
reconstructed_string = tokenizer.decode(indices)
assert string == reconstructed_string
```

vocabulary的大小为256

字节编码同样有缺陷，他的compression_ratio为1，一个token表示一个字节，这会导致转换后的整数序列过长（因为在Transformer中的上下文是有限制的，attention注意力计算是平方级的复杂度）

**Word-based tokenization**

这是一种把字符串拆分成单词的方法，使用正则表达式将字符串分割成word序列，随后为每个word分配一个token

这种方法的缺点在于无法确定vocabulary的大小，给出一个词基于你的正则表达式的规则可能会划分出新的word片段出来，如果在训练时没有选作样本，在推理时出现很罕见的词会导致模型的准确率大大降低

**Byte Pair Encoding(BPE)/bidirectional encoding**

这个算法最初被用作数据压缩，首先被引进NLP用作翻译任务（以前的工作都使用**Word-based tokenization**）

这个算法的主要思想不是应该如何事先定义好划分和转换字符串的方法和规则，而是在原始的文本上训练一个tokenizer，训练的经验直觉表明，一些常见的序列会被转化为单个的token，而一些罕见的序列会被转化为很多个tokens

为了提高训练tokenizer的效率，GPT-2首先使用的word-based tokenization的思想将原始文本分割成一些初始的文本段，随后在这些文本段上逐个运行BPE算法训练模型

首先将文本分割成段，然后将最常见的一些相似的tokens合并压缩为一个token

```python
def train_bpe(string: str, num_merges: int) -> BPETokenizerParams:
    indices = list(map(int, string.encode("utf-8")))
    merges: dict[tuple[int, int], int] = {}
    vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}

    for i in range(num_merges):
        counts = defaultdict(int)
        for index1, index2 in zip(indices, indices[1:]):
            # 统计二元组出现的个数
            counts[(index1, index2)] += 1

        # 找出出现次数最多的二元组进行合并
        pair = max(counts, key = counts.get)
        index1, index2 = pair

        # 新的index在原来的vocab上扩展
        new_index = 256 + i
        merges[pair] = new_index
        vocab[new_index] = vocab[index1] + vocab[index2]
        indices = merge(indices, pair, new_index)
```

经过上面的num_merges次循环，三次压缩了indices，最后完成编码

上面的优化方向：

encode()可以只在重要的merge中循环，而不是在所有的merges中循环

检测并保存特殊的tokens

使用pre-tokenization（例如GPT-2的tokenizer正则化）

优化实现算法的时间复杂度


## 训练分词器

在将数据喂给大模型之前需要进行分词，分词器被视为LLM的一部分，实际上有自己独立的训练，需要利用正则表达式对原始文本预处理，统计构建一套高效的词元——数字离散序列转化词表

训练分词器的步骤：

准备语料->初始化基础单元（可以省略）->统计并迭代合并->输出产物用于编码解码

### 准备语料

1.需要收集覆盖目标应用场景的多样化文本，以便训练的此表可以对下游任务有良好的泛化能力（例如多种风格的文本信息或者多种语言的文本信息

2.对原始的文本进行清洗和标准化，包含去除或屏蔽无关的元数据、修正或删除乱码与非法字符、统一字符编码为UTF-8，对重复或近重复的样本进行去重来减少训练偏移

3.对带有敏感信息或隐私的语料提前进行脱敏处理和合规的检查，明确哪些信息不可以用于训练并记录数据来源和许可

#### 命名实体识别（NER）技术解决问题

NER技术是NLP中的一项基础任务，从大量非结构化文本中自动识别应提取有特定意义的实体

具体实现的代码可以参考01-preprocess.py代码


使用数据脱敏不仅是为了保护隐私，也有助于提升下游文本建模和分词过程的稳定性，保留这些隐私信息会给分词算法引入噪声，影响其学习高频token结构时的统计效率

4.在多语言或者混合语料的场景中，需要统计各国语言的占比，适当对低资源语言进行过采样或者定向保留，避免词表被高频的语言主导，语料类型和分布不均衡会加剧低资源语言的token碎片化，增加token开销，降低任务性能

5.保留一部分未参与训练的验证语料，在训练过程中评估分词器对真实文本的编码效率和平均token长度等指标

### 初始化基本单元

1.预分词将原始文本切分成可统计、可合并的基础单元(字符、字节、Unicode片段等)，常见策略有
- 基于空格和标点的切分
- 按Unicode类别划分
- 采用字节级切分

#### 基于空格和标点切分
在句子中遇到空格或者标点可以分为独立的tokens，使用大多数预分词处理过程

In [ ]:
import re

def part(text):
    # 按照标点符号分割，同时将标点符号作为单独的token
    text = re.sub(r'([.,!?;:()"\'\[\]{}])', r' \1 ', text)
    tokens = text.split()
    return tokens

if __name__ == "__main__":
    text = "Hello, world! This is a test."
    tokens = part(text)
    print(tokens)

['Hello', ',', 'world', '!', 'This', 'is', 'a', 'test', '.']


#### Unicode 类别划分策略
按照字符的Unicode的类别（字母、数字、标点、中文、特殊字符等）自动切分，不同的类别会进入不同的token块，分类的结果是同一个token的字符类型一致，适合多语言的混合文本，可以提供一个可靠的基线切分结果

In [2]:
import unicodedata

def get_char_category(ch: str) -> str:
    """获取字符的Unicode类别"""
    cat = unicodedata.category(ch)

    # 判断中文字符
    if '\u4e00' <= ch <= '\u9fff':
        return "CJK"
    
    # 判断数字
    if ch.isdigit():
        return "DIGIT"
    
    # 判断英文字母（或者其他语言字母）
    if ch.isalpha():
        return "ALPHA"
    
    # 判断标点符号(Unicode分类中‘P’开头的类别表示标点符号)
    if cat.startswith("P"):
        return "PUNCT"
    
    # 判断其他字符
    return "OTHER"

def segment_by_unicode_category(text: str) -> list:
    if not text:
        return []
    segments = []
    # 初始化buffer和prev_type， 以第一个字符为起点
    buffer = [text[0]]
    prev_type = get_char_category(text[0])

    # 先线性扫描文本按照类别切分
    for ch in text[1:]:
        curr_type = get_char_category(ch)
        if curr_type == prev_type:
            # 类型相同存入buffer合并
            buffer.append(ch)
        else:
            # 类型不同，先将buffer中的内容作为一个segment存储，然后重置buffer和prev_type
            segments.append(("".join(buffer), prev_type))
            buffer = [ch]
            prev_type = curr_type
    
    # 最后处理剩余的buffer内容
    if buffer:
        segments.append(("".join(buffer), prev_type))

    # 提取分段后的字符串
    tokens = [seg for seg, _ in segments]
    return tokens

if __name__ == "__main__":
    s = "Hello👏👏，这是cs336的学习笔记。。。"
    tokens = segment_by_unicode_category(s)
    print("原始文本", s)
    print("分词结果", tokens)
    

原始文本 Hello👏👏，这是cs336的学习笔记。。。
分词结果 ['Hello', '👏👏', '，', '这是', 'cs', '336', '的学习笔记', '。。。']


#### 字节级的切分策略
将每个字符拆成UTF-8字节序列，不依赖语言种类字符，按照单个字节序列得到独立token

In [3]:
def tokenize_byte_level(text):
    tokens = []
    for ch in text:
        # 字符对应的UTF-8字节序列
        utf8_bytes = ch.encode('utf-8')
        hex_bytes = [f"{b:02x}" for b in utf8_bytes]

        # 转换过程
        print(f"字符: '{ch}' -> UTF-8字节: {hex_bytes}")

        tokens.extend(hex_bytes)
    return tokens

if __name__ == "__main__":
    text = "Hello, 世界!"
    tokens = tokenize_byte_level(text)
    print("原始文本:", text)
    print("Byte-level Tokens:", tokens)

字符: 'H' -> UTF-8字节: ['48']
字符: 'e' -> UTF-8字节: ['65']
字符: 'l' -> UTF-8字节: ['6c']
字符: 'l' -> UTF-8字节: ['6c']
字符: 'o' -> UTF-8字节: ['6f']
字符: ',' -> UTF-8字节: ['2c']
字符: ' ' -> UTF-8字节: ['20']
字符: '世' -> UTF-8字节: ['e4', 'b8', '96']
字符: '界' -> UTF-8字节: ['e7', '95', '8c']
字符: '!' -> UTF-8字节: ['21']
原始文本: Hello, 世界!
Byte-level Tokens: ['48', '65', '6c', '6c', '6f', '2c', '20', 'e4', 'b8', '96', 'e7', '95', '8c', '21']


英文字母和空格为1字节，中文汉字为3字节


Unicode是编码的标准，给字符分配唯一的码点，UTF-8是编码的格式，负责把码点转换为字节序列

#### 划分策略的总结
基于规则（空格和标点）切分预按Unicode类别切分的方法在以下场景中存在局限
- 文本缺乏显式分隔符
- 出现长段的同类字符（连续中文长句、拼接的代码标识符、压缩的字符串）

上面的情况中可能会导致被迫回退到更细粒度的兜底切分（字符级）

UTF-8的字节级切分策略有最强的通用性，它把任意文本统一的拆为字节序列，根本上减少了（Out of vocabulary）的问题，覆盖任意的字符集，最细粒度开始也使得它需要更多轮的共现统计与合并把零散的字节压缩未紧凑并且具有语义的token

2.大多数以空格未词边界的语言使用正则表达式按单词边界和标点初步切割，对中文、日文等不以空格为词界的语言通常采用逐字符或基于字的初始单元保证覆盖性

字节级预分词好处
- token利用率高，提高BPE合并token的自由度并尽可能合并共现频率高的单个字符，提高文本信息压缩率
- 兼容处理多种语言
- 学到更多高频的片段，减少未登录词的出现情况，模型推理更快（因为token数量少）

文本压缩率指的是一段文字转换为token后，一个token对应平均表示的文字数量，同样内容使用的token的越少，压缩率越高


3.预分词生成的基础单元序列会作为后续统计的合并输入，务必保存该序列与对应位置信息以便在训练过程反复高效更新


In [9]:
def btp_hex_list(text):
    """
    UTF-8字节级预分词，返回：
        1. tokens：每个字符的字节序列和位置信息
        2. t：所有字节的十六进制字符串列表
    """
    tokens = []
    t = []
    for idx, char in enumerate(text):
        utf8_bytes = char.encode('utf-8')
        hex_bytes = ' '.join(f"{b:02x}" for b in utf8_bytes)
        tokens.append({
            'char': char,
            'bytes': hex_bytes, # 单个字符对应的UTF-8字节序列
            'start': idx,
            'end': idx + 1
        })
        t.extend([f"{b:02x}" for b in utf8_bytes])
    return tokens, t

if __name__ == "__main__":
    text = "Hi，你好🌏"
    tokens, t = btp_hex_list(text)
    for i in tokens:
        print(i)
    print(t)

{'char': 'H', 'bytes': '48', 'start': 0, 'end': 1}
{'char': 'i', 'bytes': '69', 'start': 1, 'end': 2}
{'char': '，', 'bytes': 'ef bc 8c', 'start': 2, 'end': 3}
{'char': '你', 'bytes': 'e4 bd a0', 'start': 3, 'end': 4}
{'char': '好', 'bytes': 'e5 a5 bd', 'start': 4, 'end': 5}
{'char': '🌏', 'bytes': 'f0 9f 8c 8f', 'start': 5, 'end': 6}
['48', '69', 'ef', 'bc', '8c', 'e4', 'bd', 'a0', 'e5', 'a5', 'bd', 'f0', '9f', '8c', '8f']


### 统计和迭代更新

#### 1.子词的候选统计
遍历语料收集用于后续决策的统计信息，具体方法随不同算法变化

- BPE：统计当前的字符、字词序列中相邻对的出现频次，贪心合并出现频次最高的相邻对，迭代构建词表，决策仅仅基于频率统计
- WordPiece：评估合并或保留某些字词对语料似然即语言模型的性能贡献，选择可以显著提升语料拟合度的合并操作
- Unigram：从一个大的种子词表出发初始化每个token的概率
- SentencePiece：一个语言无关的子词分词框架，提供统一的训练和编码流程，支持多种分词算法（BPR和Unigram）。
- T5 Google提出的基于Transformer的预训练语言模型框架。将所有自然语言处理任务统一表示为“文本到文本”的形式，实现多任务统一建模，随后又有扩展的模型mT5（多语言的扩展T5）、UL2（通过混合多种去噪目标提升模型在不同下游任务的泛化能力

#### 2.迭代BPE、WordPiece、Unigram、SentencePiece四种迭代算法的分析
- BPE算法：可以借助第一步子词候选统计数据作为初始化数据，单个token合并形成新的token，在多次迭代过程中动态统计共现次数，得到新的token
- WordPiece算法：在迭代过程中动态统计当前词汇表中所有相邻子词对的出现情况，并非简单合并频率最高的对子，而是优先选择能最大提升语料整体似然的子词对，形成更有表征意义的token
  $$
  score(A,B)=\frac{P(A,B)}{P(A)\times P(B)}
  $$
  比值衡量A与B的“关联性”是否强于独立出现时的期望，score>1说明A与B结合比随机独立出现更有意义，更可能被WordPiece合并
- Unigram算法：基于子词概率语言模型，将句子的概率定义未其所有可能分词方式的概率总和，核心思想时通过迭代优化子词概率，是整个语料似然最大化，采用期望最大化（EM）方法：
  - E步（期望步）：在当前词表和子词概率下，为语料中每个句子计算最可能的分词方式（或者前n个高概率分词方案），据此估计每个子词在语料中的期望使用次数
  - M步（最大化步）：根据E步统计结果，更新子词概率，使整体语料似然最大化
  - 每次迭代中模型会剪枝概率低的token丢弃底部，收敛到预设的目标词表大小，更依赖概率建模，同时可以灵活处理不同长度子词，自然保留最能解释语料的高频段
- SentencePiece算法：独立的分词工具和实现库，从原始文本训练子词模型，直接从原始文本训练子词模型，无需显式执行预分词步骤，在预分词的初始token上应用BPE或Unigram算法，生成最后的token词表及其映射

迭代时需要保证特殊控制token在分词器迭代过程中不参与修改，确保数字映射保持固定，减少token碎片化，保护关键token的完整性，模型训练和推理的一致性
    

#### 3.通用终止条件
训练无法显著提升分词压缩效率或语言的建模质量时（词表达上限或高频合并不再明显）算法停止

#### 4.大规模语料优化
通过分布式统计和近似计数等工程手段，在保证结果稳定可复现的前提下高效处理超大规模语料（数据分散到多台机器上并行统计再合并结果

#### 5.监控和评估指标
通过token粒度、压缩率、OOV等指标评估分词器是否在“表达能力”和“效率”之间取得良好的平衡

### 输出产物用于编码和解码
#### 1.导出核心产物
训练后需要导出两个关键文件
- vocab(.json)：记录所有token和对应的id，是编码器和解码器的核心索引
- merges(.txt)：按顺序记录所有子词合并规则或概率模型，共同决定了tokenizer的编码和解码逻辑，确保编码的可逆

#### 2. 下游使用前验证和评估
- tokenizer应用于验证集后统计关键指标
- 后续要扩展加入新术语、专业词、品牌名时优先采用增量训练、加入新merges项、清理极低频token
- 扩表后做一次回归测试确保没有token分配冲突和token耗尽问题

#### 3. 版本管理和可复现性保证

## 常用分词器的实现
主要分为**字符、字节、词级、BPE分词器**

### 字符分词器

In [6]:
class CharacterTokenizer:
    def __init__(self):
        pass

    def encode(self, text):
        # 字符串编码为字符索引列表
        return [ord(ch) for ch in text]
    
    def decode(self, indices):
        # 索引列表解码为字符串
        return ''.join([chr(i) for i in indices])
    
if __name__ == "__main__":
    tokenizer = CharacterTokenizer()
    string = "Hello，世界!🐋"

    # 编码
    indices = tokenizer.encode(string)
    print("编码ID:", indices)

    # 解码
    reconstructed_string = tokenizer.decode(indices)
    print("解码:", reconstructed_string)

    # 验证解码结果与原始字符串是否一致
    assert reconstructed_string == string, "解码结果与原始字符串不一致！"

    # 计算词汇量（最大为Unicode code point + 1）
    vocabulary_size = max(indices) + 1
    print("词汇量:", vocabulary_size)

    # 简单的压缩率计算
    def get_compression_rate(text, indices):
        # 压缩率 = (原始文本字节数) / (编码索引字节数)
        import sys
        original_size = len(text.encode('utf-8'))  # 原始文本的字节数
        encoded_size = len(indices) * 4  # 编码索引的字节数（假设每个索引占4字节）
        return original_size / encoded_size if encoded_size > 0 else 0
    compression_rate = get_compression_rate(string, indices)
    print("压缩率:", compression_rate)





编码ID: [72, 101, 108, 108, 111, 65292, 19990, 30028, 33, 128011]
解码: Hello，世界!🐋
词汇量: 128012
压缩率: 0.475


最直观、最简单的分词方式，把文本拆成最小的字符单位（单个字母或汉字）
- 优势：
  - 词表小：英语只有26个字母加符号，中文只需包含几千个常用汉字
  - 无OOV问题：所有生僻的字都由基础字符组成，不会出现“未知词”
- 缺点：
  - 序列过长：变成字符后的序列很长加大LLM计算显存消耗
  - 语义稀疏：单个字符通常不具备独立语义，需要更深网络层数组合意义

### 字节分词器

In [ ]:
# 字节tokenizer
from collections import Counter
class ByteTokenizer:
    def __init__(self):
        self.vocab_size = 256

    def encode(self, text: str):
        return list(text.encode("utf-8"))
    
    def decode(self, indices):
        return bytes(indices).decode("utf-8")
    
# 字符级tokenizer
class CharTokenizer:
    def __init__(self):
        self.vocab = {}
        self.inverse_vocab = {}
    
    def encode(self, text: str):
        tokens = []
        for ch in text:
            if ch not in self.vocab:
                idx = len(self.vocab)
                self.vocab[ch] = idx
                self.inverse_vocab[idx] = ch
            tokens.append(self.vocab[ch])
        return tokens
    
    def decode(self, indices):
        return "".join(self.inverse_vocab[i] for i in indices)
    
def get_compression_ratio(text: str, token_len: int):
    input_byte_len = len(text.encode("utf-8"))
    return input_byte_len / token_len if token_len > 0 else 1
    

class BPETokenizer:
    def __init__(self, num_merges):
        self.num_merges = num_merges
        self.merges = {} # {(a,b): new_token_id}
        self.vocab_size = 256

    def get_stats(self, tokens):
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i + 1])] += 1
        return pairs
    
    def merge_tokens(self, tokens, pair, new_token):
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        return new_tokens
    
    def train(self, text: str):
        tokens = list(text.encode("utf-8"))

        for _ in range(self.num_merges):
            pairs = self.get_stats(tokens)
            if not pairs:
                break

            best_pair = max(pairs, key=pairs.get)
            new_token = self.vocab_size
            self.vocab_size += 1

            self.merges[best_pair] = new_token
            tokens = self.merge_tokens(tokens, best_pair, new_token)
    
    def encode(self, text: str):
        tokens = list(text.encode("utf-8"))

        # 按照训练好的merges顺序应用
        for pair, new_token in self.merges.items():
            tokens = self.merge_tokens(tokens, pair, new_token)
        
        return tokens
    
    def decode(self, tokens):
        # 反向展开merges（递归）
        reverse_merges = {v:k for k, v in self.merges.items()}

        def expand(token):
            if token < 256:
                return [token]
            else:
                left, right = reverse_merges[token]
                return expand(left) + expand(right)
            
        byte_seq = []
        for t in tokens:
            byte_seq.extend(expand(t))
        return bytes(byte_seq).decode("utf-8")
    
if __name__ == "__main__":
    print("=== utf-8测试 ===")
    assert bytes("a", encoding="utf-8") == b"a"
    assert bytes("🌏", encoding="utf-8") == b"\xf0\x9f\x8c\x8f"
    print("测试通过\n")
    
    text = "Hello, 🌏! 你好!"
    print("原始文本：", text)
    print("原始字节长度：", len(text.encode("utf-8")))

    # 字节级
    byte_tokenizer = ByteTokenizer()
    byte_token = byte_tokenizer.encode(text)
    byte_ratio = get_compression_ratio(text, len(byte_token))

    # 字符级
    char_tokenizer = CharTokenizer()
    char_token = char_tokenizer.encode(text)
    char_ratio = get_compression_ratio(text, len(char_token))

    # BPE
    train_text = """
    Hello world! Hello world!
    Machine learning is fun.
    你好 世界 你好 世界
    """

    bpe_tokenizer = BPETokenizer(num_merges=20)
    bpe_tokenizer.train(train_text)

    bpe_token = bpe_tokenizer.encode(text)
    bpe_ratio = get_compression_ratio(text, len(bpe_token))

    print("\n=== 对比总结 ===")
    print(f"字节级 token数：{len(byte_token)}")
    print(f"字符级 token数：{len(char_token)}")
    print(f"BPE token数：{len(bpe_token)}")

    print("\n=== 压缩率对比 ===")
    print(f"字节级：{byte_ratio:.2f}")
    print(f"字符级：{char_ratio:.2f}")
    print(f"BPE：{bpe_ratio:.2f}")

    print("\n=== 结论 ===")
    print("1. 字节级：无压缩，最稳定")
    print("2. 字符级：用utf-8字符压缩（中文/emoji更多字节）")
    print("3. BPE：通过学习高频字串，实现真正“数据驱动压缩”")

    if bpe_ratio > char_ratio:
        print("\nBPE压缩效果最好")

=== utf-8测试 ===
测试通过

原始文本： Hello, 🌏! 你好!
原始字节长度： 20

=== 对比总结 ===
字节级 token数：20
字符级 token数：13
BPE token数：11

=== 压缩率对比 ===
字节级：1.00
字符级：1.54
BPE：1.82

=== 结论 ===
1. 字节级：无压缩，最稳定
2. 字符级：用utf-8字符压缩（中文/emoji更多字节
3. BPE：通过学习高频字串，实现真正“数据驱动压缩”

BPE压缩效果最好


需要注意**字节级分词器压缩比为1**

不再维护“字符”的词表，直接维护一个大小为256的基础词表（0x00-0xFF）

现代的LLM比如GPT-4，Llama通常不使用纯字节分词，将字节作为BPE的基础单位（BBPE），彻底解决跨语言和特殊符号和emoji等的编码问题


### 词级分词器

In [14]:
import regex

# deepseek中tokenizer使用的经典正则表达式（简化）
TOKENIZER_REGEX = r"\p{L}+|\p{N}+|[^\p{L}\p{N}\s]+|\s+"

# 压缩率计算
def get_compression_ratio_word(text: str, segments):
    byte_len = len(text.encode("utf-8"))
    token_count = len(segments)
    return byte_len / token_count if token_count > 0 else 1

class WordTokenizer:
    def __init__(self, pattern=r"\w+|."):
        # 正则表达式将连续的字母数字合成一个词
        self.pattern = pattern
        self.word2id = {}
        self.id2word = {}
    
    def build_vocab(self, texts):
        vocab = set()
        for text in texts:
            segments = regex.findall(self.pattern, text)
            vocab.update(segments)
        vocab = sorted(vocab)
        self.word2id = {word: idx for idx, word in enumerate(vocab)}
        self.id2word = {idx: word for word, idx in self.word2id.items()}

    def encode(self, text):
        """
        文本 -> 字符串片段 -> token ID列表
        未登录词 UNK = -1
        """
        segments = regex.findall(self.pattern, text)
        return [self.word2id.get(seg, -1) for seg in segments], segments

    def decode(self, ids):
        """
        token ID -> 原始片段 -> 拼成字符串
        """
        return "".join(self.id2word.get(i, "<UNK>") for i in ids)

if __name__ == "__main__":

    string = "It's so supercalifragilisticexpialidocious!👋👋"
    print("原始字符串：", string)

    # 使用基础正则分词（基于空格和标点切分）
    basic_segments = regex.findall(r"\w+|.", string)
    print("基础正则分词结果：")
    print(basic_segments)

    # 使用deepseek风格正则
    segments = regex.findall(TOKENIZER_REGEX, string)
    print(f"deepseek风格分词结果：{segments}")

    # 构建词表
    tokenizer = WordTokenizer(pattern=TOKENIZER_REGEX)
    tokenizer.build_vocab([string])

    print("词表大小：", len(tokenizer.word2id))

    # 编码
    ids, segs = tokenizer.encode(string)
    print(f"编码token IDs：{ids}")

    # 字节序列
    byte_tokens = [b for b in string.encode("utf-8")]
    print(f"UTF-8字节序列：{byte_tokens}")

    print(f"编码segments：{segs}")

    # 解码
    decoded = tokenizer.decode(ids)
    print("解码结果：", decoded)

    # 压缩率
    ratio = get_compression_ratio_word(string, segs)
    print("压缩率：", ratio)

原始字符串： It's so supercalifragilisticexpialidocious!👋👋
基础正则分词结果：
['It', "'", 's', ' ', 'so', ' ', 'supercalifragilisticexpialidocious', '!', '👋', '👋']
deepseek风格分词结果：['It', "'", 's', ' ', 'so', ' ', 'supercalifragilisticexpialidocious', '!👋👋']
词表大小： 7
编码token IDs：[3, 2, 4, 0, 5, 0, 6, 1]
UTF-8字节序列：[73, 116, 39, 115, 32, 115, 111, 32, 115, 117, 112, 101, 114, 99, 97, 108, 105, 102, 114, 97, 103, 105, 108, 105, 115, 116, 105, 99, 101, 120, 112, 105, 97, 108, 105, 100, 111, 99, 105, 111, 117, 115, 33, 240, 159, 145, 139, 240, 159, 145, 139]
编码segments：['It', "'", 's', ' ', 'so', ' ', 'supercalifragilisticexpialidocious', '!👋👋']
解码结果： It's so supercalifragilisticexpialidocious!👋👋
压缩率： 6.375


词级分词器是深度学习早期（RNN）最主流的方法，基于空格（英文）或者分词算法（中文）把文本切分为具备独立语义的词

- 优势：Token保留了完整语义信息（apple直接对应一个token ID）
- 缺点：
  - 词表爆炸：不同形式的单词会视为不同ID，导致词表巨大
  - OOV问题严重：遇到没见过的人名、新造词等就标记UNK，导致信息丢失影响模型能力

### BPE 分词器

In [16]:
import regex
from collections import Counter

DEEPSEEK_REGEX = r"\p{L}+|\p{N}+|[^\p{L}\p{N}\s]+|\s+"

# 使用grapheme cluster保持emoji不被拆分
def split_graphemes(token):
    return tuple(regex.findall(r'\X', token))

# BPE训练函数
def train_bpe(texts, num_merges=50):
    """
    texts: 文本列表
    num_merges: BPE合并次数
    """
    # 构建初始vocab（字符级+</w>结束符）
    vocab = Counter()
    for text in texts:
        tokens = regex.findall(DEEPSEEK_REGEX, text)
        for token in tokens:
            chars = split_graphemes(token) + ("</w>",)  # 添加结束符
            vocab[chars] += 1
    merges = []
    for _ in range(num_merges):
        # 统计相邻的pair出现次数
        pairs = Counter()
        for token, freq in vocab.items():
            for i in range(len(token) - 1):
                pairs[(token[i], token[i + 1])] += freq
        if not pairs:
            break

        # 找出最常见的pair
        best_pair = max(pairs, key=pairs.get)
        merges.append(best_pair)

        # 合并所有vocab中的pair
        new_vocab = {}
        for word, freq in vocab.items():
            w = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and (word[i], word[i + 1]) == best_pair:
                    w.append(word[i] + word[i + 1])
                    i += 2
                else:
                    w.append(word[i])
                    i += 1
            new_vocab[tuple(w)] = freq
        vocab = new_vocab
    return merges, vocab

class BPETokenizer_1:
    def __init__(self, merges):
        self.merges = merges

    def encode_word(self, token):
        # 初始分成了字符+</w>
        word = list(split_graphemes(token)) + ["</w>"]
        # 按照merges顺序合并
        for pair in self.merges:
            i = 0
            new_word = []
            while i < len(word):
                if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                    new_word.append(word[i] + word[i + 1])
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            word = new_word
        return word
    
    def encode(self, text):
        tokens = regex.findall(DEEPSEEK_REGEX, text)
        bpe_tokens = []
        for token in tokens:
            bpe_tokens.extend(self.encode_word(token))
        return bpe_tokens
    
    def decode(self, tokens):
        # 拼接tokens并去掉结尾的</w>
        text = "".join(tokens).replace("</w>", " ")
        return text
    
if __name__ == "__main__":
    train_texts = ["这只猫🐈很可爱", "the quick brown fox jumps over the lazy 🐕‍🦺"]
    merges, vocab = train_bpe(train_texts, num_merges=20)
    print("BPE合并:", merges)
    tokenizer = BPETokenizer(merges)
    test_text = "敏捷的棕色狐狸🦊"
    encoded = tokenizer.encode(test_text)
    print("编码:", encoded)
    decoded = tokenizer.decode(encoded)
    print("解码:", decoded)


BPE合并: [(' ', '</w>'), ('t', 'h'), ('th', 'e'), ('the', '</w>'), ('这', '只'), ('这只', '猫'), ('这只猫', '</w>'), ('🐈', '</w>'), ('很', '可'), ('很可', '爱'), ('很可爱', '</w>'), ('q', 'u'), ('qu', 'i'), ('qui', 'c'), ('quic', 'k'), ('quick', '</w>'), ('b', 'r'), ('br', 'o'), ('bro', 'w'), ('brow', 'n')]
编码: [230, 149, 143, 230, 141, 183, 231, 154, 132, 230, 163, 149, 232, 137, 178, 231, 139, 144, 231, 139, 184, 240, 159, 166, 138]
解码: 敏捷的棕色狐狸🦊


目前LLM（GPT、BERT、Llama等最主流的分词算法），是一种试图在字符级和词级之间找到平衡的方法

统计语料中相邻字符对出现频率，迭代地将最频繁出现的字符对合并成一个新的token

过程：
- 初始化：把单词拆成字符序列
- 统计：计算所有相邻字符对的频率
- 合并：将频率最高的对合并成新的token
- 循环：重复上述的步骤，直到达到预设的词表大小

加入<\/w>的目的保证单词的完整性，因为BPE不会识别单词的边界，防止将一个完整的词拆成若干部分错误的和其他词部分合并

## 分析DeepSeek分词器

DeepSeek模型对代码和中英文都进行了高度的优化

### 加载分词器

In [17]:
from transformers import AutoTokenizer
MODEL_NAME = "deepseek-ai/deepseek-coder-6.7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"成功加载模型：{MODEL_NAME}的分词器")
print(f"分词器词表大小：{len(tokenizer.get_vocab())}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

成功加载模型：deepseek-ai/deepseek-coder-6.7b-instruct的分词器
分词器词表大小：32022


### DeepSeek分词器的处理逻辑



In [18]:
chinese_text = "hello world!注意力机制是AI的核心技术。 🚀 🚀"
# 编码
encoded_ids = tokenizer.encode(chinese_text, add_special_tokens=False)
# 解码回Token字符串 (用于观察子词)
tokens = tokenizer.convert_ids_to_tokens(encoded_ids)
text = tokenizer.decode(encoded_ids)
print(f"\n原文: {chinese_text}")
print(f"原始子编码: {tokens}")
print(f"最终解码: {text}")
print(f"IDs:{encoded_ids}")


原文: hello world!注意力机制是AI的核心技术。 🚀 🚀
原始子编码: ['hello', 'Ġworld', '!', 'æ³¨æĦı', 'åĬĽ', 'æľºåĪ¶', 'æĺ¯', 'AI', 'çļĦ', 'æł¸å¿ĥ', 'æĬĢæľ¯', 'ãĢĤ', 'ĠðŁ', 'ļ', 'Ģ', 'ĠðŁ', 'ļ', 'Ģ']
最终解码: hello world!注意力机制是AI的核心技术。 🚀 🚀
IDs:[31702, 1835, 0, 7421, 1405, 16737, 502, 26888, 337, 12282, 4645, 397, 12394, 235, 209, 12394, 235, 209]


这里子编码中的乱码并非词表映射，源自大模型词表的设计机制导致，其中英文基础词汇和常用的子词已经收录词表，但是没有将全部汉字逐一收录，采用字节级编码（UTF-8）对中文兜底表示，所以中文会显示乱码，除了使用decode方法还原，还可以训练BPE

In [1]:
import regex as re
from collections import Counter
from typing import List, Tuple, Dict, Iterable
import json
import base64

DEEPSEEK_REGEX = r"\p{L}+|\p{N}+|[^\p{L}\p{N}\s]+|\s+"

# 预分词和字节处理
def pretokenize(text: str):
    # 使用deepseek风格的正则表达式进行预分词
    segments = re.findall(DEEPSEEK_REGEX, text)
    return segments

def bytes2tokens(b: bytes):
    """
    将UTF-8字节序列转化为latin1可表示的token列表
    每个字节0-255都可以被latin1映射到字符
    """
    return [bytes([x]).decode('latin1') for x in b]

def tokens2bytes(tokens):
    # 将latin1 token列表重新转回原始bytes
    return b''.join(t.encode('latin1') for t in tokens)

# BPExunl
def build_corpus(texts):
    """
    构建byte-level的语料
    预分词->UTF-8编码 ->分解为单字节 -> 作为初始token序列
    """
    corpus = []
    for text in texts:
        for chunk in pretokenize(text):
            corpus.append(bytes2tokens(chunk.encode('utf-8')))
    return corpus

def pair_freq(corpus: List[List[str]]) -> Counter:
    """
    统计所有token序列中相邻token对的频率
    """
    pairs = Counter()
    for word in corpus:
        for i in range(len(word) - 1):
            pairs[(word[i], word[i + 1])] += 1
    return pairs

def merge_pair(word: List[str], pair: Tuple[str, str]):
    """
    在一个token序列中合并指定的pair
    """
    a, b = pair
    merged = []
    i = 0
    while i < len(word):
        if i < len(word) - 1 and word[i] == a and word[i + 1] == b:
            merged.append(a + b)
            i += 2
        else:
            merged.append(word[i])
            i += 1
    return merged

def train_bpe(texts: Iterable[str], vocab_size: int=5000, num_merges: int=None) -> Tuple[List[Tuple[str, str]], List[str]]:
    """
    训练字节级BPE
    """
    corpus = build_corpus(texts)
    base_tokens = [bytes([i]).decode('latin1') for i in range(256)]  # 初始token为单字节
    merges: List[Tuple[str, str]] = []
    merged_set = set()
    cur_vocab_size = 256

    # 若未指定合并次数，由target vocab决定
    merge_steps = num_merges or (vocab_size - 256)

    for _ in range(merge_steps):
        pfreq = pair_freq(corpus)
        if not pfreq:
            break

        # 找到频率最高的pair
        best_pair, _ = pfreq.most_common(1)[0]
        if cur_vocab_size >= vocab_size:
            break

        merges.append(best_pair)

        # 对整个语料进行合并
        corpus = [merge_pair(word, best_pair) for word in corpus]

        # 将新token计入词表
        merged_set.add(best_pair[0] + best_pair[1])
        cur_vocab_size += 1
    
    # 追加特殊的token
    special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]

    # vocab = 特殊token + 256 byte token + BPE合并的新token
    vocab_tokens = special_tokens + base_tokens + sorted(merged_set)

    return merges, vocab_tokens

class DeepSeekV3Tokenizer:
    def __init__(self, merges: List[Tuple[str, str]], vocab_tokens: List[str]):
        self.merges = merges
        self.vocab_tokens = vocab_tokens
        self.token2id = {token: idx for idx, token in enumerate(vocab_tokens)}
        self.id2token = {idx: token for token, idx in self.token2id.items()}

        self.ranks = {pair: idx for idx, pair in enumerate(merges)}

        self.pad_token = "<pad>"
        self.bos_token = "<bos>"
        self.eos_token = "<eos>"
        self.unk_token = "<unk>"

    def encode_chunk(self, chunk: str) -> List[str]:
        """
        对一个预分词做BPE编码：
        - 转字节token
        - 逐步应用merges
        - 处理OOV：未知token拆回字节或标记为<unk>
        """
        tokens = bytes2tokens(chunk.encode('utf-8'))

        # 应用PE并规则
        for pair in self.merges:
            new_tokens = []
            i = 0
            a,b = pair
            while i < len(tokens):
                if i<len(tokens)-1 and tokens[i]==a and tokens[i+1]==b:
                    new_tokens.append(a+b)
                    i+=2
                else:
                    new_tokens.append(tokens[i])
                    i+=1
            tokens = new_tokens

        # OOV token拆回字节
        out = []
        for t in tokens:
            if t in self.token2id:
                out.append(t)
            else:
                # 拆分成字节token，如果字节token也不在词表 → <unk>
                out.extend([ch if ch in self.token2id else self.unk_token for ch in t])
        return out

    def encode(self, text: str, add_bos=False, add_eos=False, print_chunks=False):
        """
        编码完整文本：
        - 先预分词
        - 再逐chunk编码
        - 可选打印中间过程
        """
        ids = []

        if add_bos:
            ids.append(self.token2id[self.bos_token])
            if print_chunks: print(f"[Special] <bos> -> {self.token2id[self.bos_token]}")

        for chunk in pretokenize(text):
            toks = self.encode_chunk(chunk)
            chunk_ids = [self.token2id.get(t, self.token2id[self.unk_token]) for t in toks]

            if print_chunks:
                readable = []
                for t in toks:
                    try:
                        # 尝试恢复utf-8
                        r = tokens2bytes([t]).decode('utf-8', errors='ignore')
                        readable.append(r if r else t.encode('latin1').hex())
                    except:
                        readable.append(t.encode('latin1').hex())

                print(f"[Chunk] \"{chunk}\" -> {readable} -> IDs: {chunk_ids}")

            ids.extend(chunk_ids)

        if add_eos:
            ids.append(self.token2id[self.eos_token])
            if print_chunks: print(f"[Special] <eos> -> {self.token2id[self.eos_token]}")
        return ids

    def decode(self, ids: Iterable[int]):
        """
        将ID序列还原为utf-8文本：
        """
        byte_seq = bytearray()
        for i in ids:
            tok = self.id2token.get(i, self.unk_token)
            if tok in {self.pad_token, self.bos_token, self.eos_token}:
                continue
            byte_seq.extend(tokens2bytes(list(tok)))
        return byte_seq.decode('utf-8', errors='replace')

    def save(self, vocab_path: str, merges_path: str):

        # 保存vocab（token2id）
        with open(vocab_path, 'w', encoding='utf-8') as f:
            json.dump(self.token2id, f, ensure_ascii=False, indent=2)

        # 保存merges：每个token用base64
        merges_b64 = []
        for a, b in self.merges:
            a_bytes = a.encode('latin1')
            b_bytes = b.encode('latin1')
            merges_b64.append((
                base64.b64encode(a_bytes).decode('ascii'),
                base64.b64encode(b_bytes).decode('ascii')
            ))

        with open(merges_path, 'w', encoding='utf-8') as f:
            json.dump(merges_b64, f, ensure_ascii=False, indent=2)

    @classmethod
    def load(cls, vocab_path: str, merges_path: str):

        # 加载vocab
        with open(vocab_path, 'r', encoding='utf-8') as f:
            token2id = json.load(f)
        vocab_tokens = [None] * (max(token2id.values()) + 1)
        for tok, idx in token2id.items():
            vocab_tokens[idx] = tok

        # 加载merges（base64 → bytes → latin1）
        with open(merges_path, 'r', encoding='utf-8') as f:
            merges_b64 = json.load(f)

        merges = []
        for a_b64, b_b64 in merges_b64:
            a = base64.b64decode(a_b64).decode('latin1')
            b = base64.b64decode(b_b64).decode('latin1')
            merges.append((a, b))
        return cls(merges, vocab_tokens)


# 提供训练函数
def train_tokenizer(texts, vocab_size=5000, num_merges=None):
    merges, vocab_tokens = train_bpe(texts, vocab_size=vocab_size, num_merges=num_merges)
    return DeepSeekV3Tokenizer(merges, vocab_tokens)

if __name__ == "__main__":
    texts = [
        "Transformer是AI的核心技术。",
        "DeepSeek分词器支持中文、英文、emoji等多语言。",
        "Hello, 世界! 🌍🚀",
    ]

    print("训练 Tokenizer (vocab_size=1024)")
    tokenizer = train_tokenizer(texts, vocab_size=1024)
    print(f"完成训练，词表大小: {len(tokenizer.vocab_tokens)}")
    print("-"*50)

    txt = "注意力机制是AI的核心技术。 🚀 🚀"
    print(f"编码文本: {txt}")
    ids = tokenizer.encode(txt, add_bos=True, add_eos=True, print_chunks=True)

    print("-"*50)
    print("Token ID:", ids)
    decoded = tokenizer.decode(ids)
    print("解码结果:", decoded)
    print("是否可逆:", decoded == txt)

训练 Tokenizer (vocab_size=1024)
完成训练，词表大小: 352
--------------------------------------------------
编码文本: 注意力机制是AI的核心技术。 🚀 🚀
[Special] <bos> -> 2
[Chunk] "注意力机制是AI的核心技术" -> ['e6', 'b3', 'a8', 'e6', '84', '8f', 'e5', '8a', '9b', 'e6', '9c', 'ba', 'e5', '88', 'b6', 'e6', '98', 'af', 'A', 'I', 'e7', '9a', '84', 'e6', 'a0', 'b8', 'e5', 'bf', '83', 'e6', '8a', '80', 'e6', '9c', 'af'] -> IDs: [234, 183, 172, 234, 136, 147, 233, 142, 159, 234, 160, 190, 233, 140, 186, 234, 156, 179, 69, 77, 235, 158, 136, 234, 164, 188, 233, 195, 135, 234, 142, 132, 234, 160, 179]
[Chunk] "。" -> ['。'] -> IDs: [334]
[Chunk] " " -> [' '] -> IDs: [36]
[Chunk] "🚀" -> ['f09f', '9a', '80'] -> IDs: [346, 158, 132]
[Chunk] " " -> [' '] -> IDs: [36]
[Chunk] "🚀" -> ['f09f', '9a', '80'] -> IDs: [346, 158, 132]
[Special] <eos> -> 3
--------------------------------------------------
Token ID: [2, 234, 183, 172, 234, 136, 147, 233, 142, 159, 234, 160, 190, 233, 140, 186, 234, 156, 179, 69, 77, 235, 158, 136, 234, 164, 188, 23

依据分词算法将文本离散化为最小语义单元token，构建其与全局唯一数值ID及底层代码的确定性映射，相同的字符在文中指向一致的ID和编码序列，确保了特征表征的稳定性

分词器中的token<->id映射只描述token的内容，不包含任何其在句子中的位置信息，BPE或其他基于概率和统计的分词算法本质都是依据语料中的共现频率或概率分布，并不理解句子语义

#### DeepSeek使用Latin1编解码原因
DeepSeek分词流程中最终得到数字化的token，在BPE分词器训练时要按“字符”进行操作。由于UTF-8编解码中汉字和emoji为多字节的字符，拆分为单字节时序列不完整导致会报错，进而导致信息丢失，latin-1为单字节编码，将0-255的每个字节映射到一个Unicode字符，保证BPE或其他的子词算法把字节当作字符合并，保证编码器阶段信息完整